# Location selection

In [114]:
from desdeo.problem import Constant, Variable, Problem, Objective, VariableTypeEnum
import numpy as np

# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Model inputs

In [115]:
# Mininum expected attendance to be worth visiting 
min_att = 15

# Cost constants
# The current gas costs ($/gallon)
raw_dollars_per_gallon = 3.00

# The efficiency of the vehicle (miles/gallon)
raw_mpg = 6.0
# How long the event is (hours)
hours_per_event = 4
driver_salary_per_hour = 19
raw_driver_cost_per_trip = hours_per_event * driver_salary_per_hour

# Food desert threshold (miles) (TODO this should be in minutes)
raw_food_desert_threshold = 15



## Load and process constants


In [116]:
from slugify import slugify
import pandas as pd

def no_nan(val):
    if pd.isna(val):
        return ""
    else:
        return str(val)

home = "Ada"
# Read the adjacency matrix 
adjacencies = pd.read_csv("adjacencyMatrix.csv", index_col=0)
dist2home = adjacencies.loc[:,[home]].rename(columns={home:"dist2home"})
dist2home.index.name = "city"

# Read the cities 
cities = pd.read_csv("cities.csv")
sites = pd.read_csv("sites.csv")

# Create event table
events = pd.merge(cities,sites,on="city")
events.loc[:,"expectedAttendance"] = (events.loc[:,"pop"] * events.loc[:,"attendanceRate"]).astype(int)

events.loc[:, "event_id"] = events.apply(lambda row: slugify(f'{row["city"]} {row["site"]} {no_nan(row["event"])}'), axis=1)

# Add the distance to home for each event 
events = pd.merge(events, dist2home, on="city")

events

,city,lat,long,pop,site,event,attendanceRate,expectedAttendance,event_id,dist2home
0,Ada,40.768056,-83.825278,5334,Public Library,NaN,0.002,10,ada-public-library,0.0
1,Lima,40.746389,-84.123333,35579,Mercy Health Thrift,NaN,0.002,71,lima-mercy-health-thrift,16.2
2,Lima,40.746389,-84.123333,35579,Habitat For Humanity,NaN,0.002,71,lima-habitat-for-humanity,16.2
3,Lima,40.746389,-84.123333,35579,Our Daily Bread,NaN,0.002,71,lima-our-daily-bread,16.2
4,Lima,40.746389,-84.123333,35579,St. Mark’s Methodist,Community Meal,0.002,71,lima-st-marks-methodist-community-meal,16.2
5,Lima,40.746389,-84.123333,35579,Christian Corner Community Center,NaN,0.002,71,lima-christian-corner-community-center,16.2
6,Kenton,40.646667,-83.622500,7947,Seton Hall,NaN,0.002,15,kenton-seton-hall,15.5
7,Kenton,40.646667,-83.622500,7947,Hardincrest,NaN,0.002,15,kenton-hardincrest,15.5
8,Kenton,40.646667,-83.622500,7947,YMCA,NaN,0.002,15,kenton-ymca,15.5
9,Delphos,40.861111,-84.350000,7117,Public Library,NaN,0.002,14,delphos-public-library,31.8


### Close cities

In [117]:
close_cities = adjacencies < raw_food_desert_threshold
#close_cities = close_cities.sum(axis=1) - 1
close_cities

,Ada,Lima,Kenton,Delphos,Bluffton,Spencerville,Elida,Forest,Alger,Dunkirk,Mount Victory,Cairo,St. Marys
Ada,True,False,False,False,True,False,False,False,True,True,False,False,False
Lima,False,True,False,True,False,True,True,False,False,False,False,True,False
Kenton,False,False,True,False,False,False,False,True,True,True,True,False,False
Delphos,False,True,False,True,False,True,True,False,False,False,False,True,False
Bluffton,True,False,False,False,True,False,False,False,False,False,False,True,False
Spencerville,False,True,False,True,False,True,True,False,False,False,False,False,True
Elida,False,True,False,True,False,True,True,False,False,False,False,True,False
Forest,False,False,True,False,False,False,False,True,False,True,False,False,False
Alger,True,False,True,False,False,False,False,False,True,False,False,False,False
Dunkirk,True,False,True,False,False,False,False,True,False,True,False,False,False


# Constants

In [118]:
event_count = events.shape[0]

# Expected attenance 
raw_attendance = events.loc[:,"expectedAttendance"] 
raw_over_attendance = raw_attendance < min_att

# Expected attendance 
exp_att = []
# Over staffed events (ose)
ose = []

d2h = []
for e in range(event_count): 
    exp_att.append(Constant(name=f"Expected attendance #{e}", 
                        symbol=f"exp_att_{e}", 
                        type="int", 
                        value=raw_attendance[e]
                        ))

    ose.append(Constant(name=f"Over attendance #{e}", 
                        symbol=f"ose_{e}", 
                        type="binary",
                        value=raw_over_attendance[e]))

    city = events.loc[e,"city"]
    d2h.append(Constant(name=f"Distance to home #{e}", 
                              symbol=f"d2h_{e}", 
                              type="real", 
                              value=dist2home.loc[city]
                              ))



mpg = Constant(name="Miles per gallon (miles/gallon)", 
               symbol="mpg", 
               type="real",
               value=raw_mpg)

dpg = Constant(name="Dollars per gallon ($/gallon)", 
               symbol="dpg", 
               type="real",
               value=raw_dollars_per_gallon)

driver_cost_per_trip = Constant(name="Driver cost per trip ($)", 
                                symbol="dcpt", 
                                type="real", 
                                value=raw_driver_cost_per_trip)


food_desert_threshold = Constant(name="Food desert threshold (miles)", 
                                 symbol="fdt", 
                                 type="real",
                                 value=raw_food_desert_threshold
                                 )



## Variables
Generate one variable per event

In [150]:
evt_visited = []
for e in range(event_count):
  evt_visited.append(Variable(
    name=events.loc[e, "event_id"],
    symbol=f"evt_visit_{e}",
    variable_type=VariableTypeEnum.integer,
    lowerbound=0,
    upperbound=1))

## Objectives

### Build the objective expression strings

In [151]:
# total patients served
obj_func_total_patients = " + ".join([f"evt_visit_{e} * exp_att_{e}" for e in range(event_count)])
display(obj_func_total_patients)


'evt_visit_0 * exp_att_0 + evt_visit_1 * exp_att_1 + evt_visit_2 * exp_att_2 + evt_visit_3 * exp_att_3 + evt_visit_4 * exp_att_4 + evt_visit_5 * exp_att_5 + evt_visit_6 * exp_att_6 + evt_visit_7 * exp_att_7 + evt_visit_8 * exp_att_8 + evt_visit_9 * exp_att_9 + evt_visit_10 * exp_att_10 + evt_visit_11 * exp_att_11 + evt_visit_12 * exp_att_12 + evt_visit_13 * exp_att_13 + evt_visit_14 * exp_att_14 + evt_visit_15 * exp_att_15 + evt_visit_16 * exp_att_16 + evt_visit_17 * exp_att_17'

In [152]:

# ose = over staffed events
obj_func_ose = " + ".join([f"evt_visit_{e} * ose_{e}" for e in range(event_count)])
display(obj_func_ose)


'evt_visit_0 * ose_0 + evt_visit_1 * ose_1 + evt_visit_2 * ose_2 + evt_visit_3 * ose_3 + evt_visit_4 * ose_4 + evt_visit_5 * ose_5 + evt_visit_6 * ose_6 + evt_visit_7 * ose_7 + evt_visit_8 * ose_8 + evt_visit_9 * ose_9 + evt_visit_10 * ose_10 + evt_visit_11 * ose_11 + evt_visit_12 * ose_12 + evt_visit_13 * ose_13 + evt_visit_14 * ose_14 + evt_visit_15 * ose_15 + evt_visit_16 * ose_16 + evt_visit_17 * ose_17'

In [153]:
# Total costs
total_distance = " + ".join([f"evt_visit_{e} * d2h_{e}" for e in range(event_count)])
print("Total distance:")
display(total_distance)

print("Total gas cost")
total_gas_costs = f"(({total_distance})/mpg)*dpg"
display(total_gas_costs)

# TODO I can represent the drive time of the driver here once I have the drive distance matrix
print("Total driver cost")
total_driver_cost = " + ".join([f"evt_visit_{e}" for e in range(event_count)])
total_driver_cost = f"({total_driver_cost})*dcpt"

display(total_driver_cost)

print("Total costs")
obj_total_cost = f"{total_gas_costs} + {total_driver_cost}"
display(obj_total_cost)


Total distance:


'evt_visit_0 * d2h_0 + evt_visit_1 * d2h_1 + evt_visit_2 * d2h_2 + evt_visit_3 * d2h_3 + evt_visit_4 * d2h_4 + evt_visit_5 * d2h_5 + evt_visit_6 * d2h_6 + evt_visit_7 * d2h_7 + evt_visit_8 * d2h_8 + evt_visit_9 * d2h_9 + evt_visit_10 * d2h_10 + evt_visit_11 * d2h_11 + evt_visit_12 * d2h_12 + evt_visit_13 * d2h_13 + evt_visit_14 * d2h_14 + evt_visit_15 * d2h_15 + evt_visit_16 * d2h_16 + evt_visit_17 * d2h_17'

Total gas cost


'((evt_visit_0 * d2h_0 + evt_visit_1 * d2h_1 + evt_visit_2 * d2h_2 + evt_visit_3 * d2h_3 + evt_visit_4 * d2h_4 + evt_visit_5 * d2h_5 + evt_visit_6 * d2h_6 + evt_visit_7 * d2h_7 + evt_visit_8 * d2h_8 + evt_visit_9 * d2h_9 + evt_visit_10 * d2h_10 + evt_visit_11 * d2h_11 + evt_visit_12 * d2h_12 + evt_visit_13 * d2h_13 + evt_visit_14 * d2h_14 + evt_visit_15 * d2h_15 + evt_visit_16 * d2h_16 + evt_visit_17 * d2h_17)/mpg)*dpg'

Total driver cost


'(evt_visit_0 + evt_visit_1 + evt_visit_2 + evt_visit_3 + evt_visit_4 + evt_visit_5 + evt_visit_6 + evt_visit_7 + evt_visit_8 + evt_visit_9 + evt_visit_10 + evt_visit_11 + evt_visit_12 + evt_visit_13 + evt_visit_14 + evt_visit_15 + evt_visit_16 + evt_visit_17)*dcpt'

Total costs


'((evt_visit_0 * d2h_0 + evt_visit_1 * d2h_1 + evt_visit_2 * d2h_2 + evt_visit_3 * d2h_3 + evt_visit_4 * d2h_4 + evt_visit_5 * d2h_5 + evt_visit_6 * d2h_6 + evt_visit_7 * d2h_7 + evt_visit_8 * d2h_8 + evt_visit_9 * d2h_9 + evt_visit_10 * d2h_10 + evt_visit_11 * d2h_11 + evt_visit_12 * d2h_12 + evt_visit_13 * d2h_13 + evt_visit_14 * d2h_14 + evt_visit_15 * d2h_15 + evt_visit_16 * d2h_16 + evt_visit_17 * d2h_17)/mpg)*dpg + (evt_visit_0 + evt_visit_1 + evt_visit_2 + evt_visit_3 + evt_visit_4 + evt_visit_5 + evt_visit_6 + evt_visit_7 + evt_visit_8 + evt_visit_9 + evt_visit_10 + evt_visit_11 + evt_visit_12 + evt_visit_13 + evt_visit_14 + evt_visit_15 + evt_visit_16 + evt_visit_17)*dcpt'

### Create the objective objects

In [154]:

# Total patients visited 
total_patients = Objective(
    name = "Maximize total patients visited",
    symbol = "f_1", 
    maximize = True,
    is_twice_differentiable=True,
    func = obj_func_total_patients
)

# Overstaffed events
over_staffed_events = Objective(
    name = "Minimize the number of overstaffed events",
    symbol = "f_2",
    maximize = False,
    is_twice_differentiable=True,
    func = obj_func_ose
)

# Total costs
costs = Objective(
    name = "Minimize the total costs",
    symbol = "f_3",
    maximize = False,
    is_twice_differentiable=True,
    func = obj_total_cost
)

# Communities served



## Problem 


In [155]:
 

prob = Problem(
        name="Simple site selection",
        description="Simple implementation of the site selection problem",
        type="linear",
        constants=exp_att +  ose + d2h + [mpg, dpg, driver_cost_per_trip, food_desert_threshold],
        variables=evt_visited,
        objectives=[total_patients, over_staffed_events, costs],
        constraints=[]
    )


## Ideal/nadir


In [157]:
# f_1 ideal is seeing all patients, nadir is seeing no patients
all_patients = int(np.sum(events.loc[:,"expectedAttendance"]))

# f_2 ideal is having no over staffed events
# f_2 naird is having visiting all locations with over staffed events
all_ose = int(np.sum(raw_over_attendance))

# f_3 ideal is no costs
# f_3 nadir is going to every site

# How many miles are driven/gas costs
max_dist = events.loc[:,"dist2home"].sum()
total_gas_cost = (max_dist / raw_mpg) * raw_dollars_per_gallon
driver_cost = events.shape[0]*raw_driver_cost_per_trip
max_costs = float(driver_cost + total_gas_cost)


prob = prob.update_ideal_and_nadir(
    new_ideal={
        "f_1": all_patients,
        "f_2": 0,
        "f_3": 0
        }, 
    new_nadir={
        "f_1": 0,
        "f_2": all_ose,
        "f_3": max_costs
        }
    )

print(f"Ideal values: {prob.get_ideal_point()}")
print(f"Nadir values: {prob.get_nadir_point()}")

Ideal values: {'f_1': 441, 'f_2': 0, 'f_3': 0}
Nadir values: {'f_1': 0, 'f_2': 10, 'f_3': 1506.95}


## Solve (with NSGA-III)!

In [126]:
from desdeo.emo.methods.EAs import nsga3_mixed_integer


#solver, publisher = nsga3_mixed_integer(problem=prob)
#
#result = solver()



### Clean up NSGA-III results 

In [127]:
pf_obj = (result.outputs.to_pandas()).astype(int).loc[:,["f_1", "f_2", "f_3"]]
pf_dec = (result.solutions.to_pandas()).astype(bool)

results = pd.DataFrame(pf_obj.values, 
                       columns=["Total Patients Served", "Number of over staffed events", "Total Costs"])

# Build string lists for the events visited
pf_events_visited = []
for (e, ev) in pf_dec.iterrows(): 
    events_visited = ev.values
    pf_events_visited.append("\n".join(events.loc[events_visited, "event_id"].values))


results.loc[:,"events_visited"] = pf_events_visited

results = results.drop_duplicates()

results

,Total Patients Served,Number of over staffed events,Total Costs,events_visited
0,372,2,578,ada-public-library\nlima-mercy-health-thrift\n...
1,387,2,662,ada-public-library\nlima-mercy-health-thrift\n...
2,284,0,336,lima-mercy-health-thrift\nlima-our-daily-bread...
3,152,1,244,ada-public-library\nlima-st-marks-methodist-co...
4,424,2,839,ada-public-library\nlima-mercy-health-thrift\n...
...,...,...,...,...
475,157,0,251,lima-habitat-for-humanity\nlima-christian-corn...
485,71,0,84,lima-our-daily-bread
486,223,1,328,ada-public-library\nlima-habitat-for-humanity\...
487,142,0,168,lima-habitat-for-humanity\nlima-christian-corn...


## Plot plot plot

In [128]:
%matplotlib qt

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

x = results.loc[:,"Total Patients Served"].values
y = results.loc[:,"Number of over staffed events"].values
z = results.loc[:,"Total Costs"].values

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(x, y, z,
           linewidths=1, alpha=.7,
           edgecolor='k',
           s = 200)

ax.set_xlabel("Total patients served")
ax.set_ylabel("Number of over staffed events")
ax.set_zlabel("Total costs")

#cbar = plt.colorbar(scatter, ax=ax, pad=0.1)
#cbar.set_label('Total Travel Costs ($)')  # Label for the color bar

plt.show()

## Solve! (with reference point method)

In [ ]:
from desdeo.mcdm.reference_point_method import rpm_solve_solutions

reference_point = {"f_1": 200, "f_2": 2, "f_3": 1000}

results = rpm_solve_solutions(prob, reference_point=reference_point)


WARNING (W1001): Setting Var 'evt_visit_0' to a value `0.5` (float) not in
domain NonNegativeIntegers.
    See also https://pyomo.readthedocs.io/en/stable/errors.html#w1001
WARNING (W1001): Setting Var 'evt_visit_1' to a value `0.5` (float) not in
domain NonNegativeIntegers.
    See also https://pyomo.readthedocs.io/en/stable/errors.html#w1001
WARNING (W1001): Setting Var 'evt_visit_2' to a value `0.5` (float) not in
domain NonNegativeIntegers.
    See also https://pyomo.readthedocs.io/en/stable/errors.html#w1001
WARNING (W1001): Setting Var 'evt_visit_3' to a value `0.5` (float) not in
domain NonNegativeIntegers.
    See also https://pyomo.readthedocs.io/en/stable/errors.html#w1001
WARNING (W1001): Setting Var 'evt_visit_4' to a value `0.5` (float) not in
domain NonNegativeIntegers.
    See also https://pyomo.readthedocs.io/en/stable/errors.html#w1001
WARNING (W1001): Setting Var 'evt_visit_5' to a value `0.5` (float) not in
domain NonNegativeIntegers.
    See also https://pyomo.readt

In [159]:
for i, result in enumerate(results):
    print(f"Solution {i+1}:")
    print(f"Objective function values \t\t {result.optimal_objectives}")
    print(f"Decision variable values \t\t {result.optimal_variables}")
    print(f"Constraint values \t\t\t {result.constraint_values}")
    print("---")

results


Solution 1:
Objective function values 		 {'f_1': 355.0, 'f_2': 0.0, 'f_3': 420.5}
Decision variable values 		 {'evt_visit_0': 0.0, 'evt_visit_1': 1.0, 'evt_visit_2': 1.0, 'evt_visit_3': 1.0, 'evt_visit_4': 1.0, 'evt_visit_5': 1.0, 'evt_visit_6': 0.0, 'evt_visit_7': 0.0, 'evt_visit_8': 0.0, 'evt_visit_9': 0.0, 'evt_visit_10': 0.0, 'evt_visit_11': 0.0, 'evt_visit_12': 0.0, 'evt_visit_13': 0.0, 'evt_visit_14': 0.0, 'evt_visit_15': 0.0, 'evt_visit_16': 0.0, 'evt_visit_17': 0.0, '_alpha': -0.19999998999000201}
Constraint values 			 {'f_1_con': -0.1514739321154993, 'f_2_con': 9.989999993909038e-09, 'f_3_con': -0.18455158744484873}
---
Solution 2:
Objective function values 		 {'f_1': 441.0, 'f_2': 10.0, 'f_3': 1506.95}
Decision variable values 		 {'evt_visit_0': 1.0, 'evt_visit_1': 1.0, 'evt_visit_2': 1.0, 'evt_visit_3': 1.0, 'evt_visit_4': 1.0, 'evt_visit_5': 1.0, 'evt_visit_6': 1.0, 'evt_visit_7': 1.0, 'evt_visit_8': 1.0, 'evt_visit_9': 1.0, 'evt_visit_10': 1.0, 'evt_visit_11': 1.0, 'evt_vi

[SolverResults(optimal_variables={'evt_visit_0': 0.0, 'evt_visit_1': 1.0, 'evt_visit_2': 1.0, 'evt_visit_3': 1.0, 'evt_visit_4': 1.0, 'evt_visit_5': 1.0, 'evt_visit_6': 0.0, 'evt_visit_7': 0.0, 'evt_visit_8': 0.0, 'evt_visit_9': 0.0, 'evt_visit_10': 0.0, 'evt_visit_11': 0.0, 'evt_visit_12': 0.0, 'evt_visit_13': 0.0, 'evt_visit_14': 0.0, 'evt_visit_15': 0.0, 'evt_visit_16': 0.0, 'evt_visit_17': 0.0, '_alpha': -0.19999998999000201}, optimal_objectives={'f_1': 355.0, 'f_2': 0.0, 'f_3': 420.5}, constraint_values={'f_1_con': -0.1514739321154993, 'f_2_con': 9.989999993909038e-09, 'f_3_con': -0.18455158744484873}, extra_func_values=None, scalarization_values={'_asf': -0.20000051593821658}, success=True, message="Pyomo solver status is: 'ok', with termination condition: 'optimal'."),
 SolverResults(optimal_variables={'evt_visit_0': 1.0, 'evt_visit_1': 1.0, 'evt_visit_2': 1.0, 'evt_visit_3': 1.0, 'evt_visit_4': 1.0, 'evt_visit_5': 1.0, 'evt_visit_6': 1.0, 'evt_visit_7': 1.0, 'evt_visit_8': 1.0,